# Masking Inspector — Llama 3.1

Compare **legacy** (`instruction_template` + `response_template`) vs **span-based** (`assistant_turn` with `end_of_turn_template`) masking using the Llama 3.1 tokenizer.

Key difference from Qwen: Llama formats `tool` role as `<|start_header_id|>ipython<|end_header_id|>`, **not** as a user message. So the legacy collator's `instruction_template` won't match tool results → they get incorrectly trained on.

In [1]:
import json
import warnings

from transformers import AutoTokenizer

from oumi.core.collators.text_completions_collator_with_padding import (
    TextCompletionsCollatorWithPadding,
)

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
# Llama has no pad token by default — set one
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Loaded tokenizer: {MODEL}")
print(
    f"Vocab size: {tokenizer.vocab_size}, pad={tokenizer.pad_token_id}, eos={tokenizer.eos_token_id}"
)

Loaded tokenizer: meta-llama/Llama-3.1-8B-Instruct
Vocab size: 128000, pad=128009, eos=128009


In [2]:
# Show how Llama formats each role
demo = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Call a tool"},
    {"role": "assistant", "content": '<tool_call>{"name": "do_thing"}</tool_call>'},
    {"role": "tool", "content": "result=42"},
    {"role": "assistant", "content": "The answer is 42."},
]
print(tokenizer.apply_chat_template(demo, tokenize=False, add_generation_prompt=False))
print()
print("Notice: tool role → <|start_header_id|>ipython<|end_header_id|>")
print(
    "Legacy instruction_template '<|start_header_id|>user<|end_header_id|>' does NOT match this."
)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are helpful.<|eot_id|><|start_header_id|>user<|end_header_id|>

Call a tool<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<tool_call>{"name": "do_thing"}</tool_call><|eot_id|><|start_header_id|>ipython<|end_header_id|>

"result=42"<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The answer is 42.<|eot_id|>

Notice: tool role → <|start_header_id|>ipython<|end_header_id|>
Legacy instruction_template '<|start_header_id|>user<|end_header_id|>' does NOT match this.


In [5]:
def make_old_collator(tokenizer):
    """Legacy collator: instruction_template + response_template (old Llama defaults)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        return TextCompletionsCollatorWithPadding(
            tokenizer=tokenizer,
            train_target="_legacy_instruction_response",
            instruction_template="<|start_header_id|>user<|end_header_id|>\n\n",
            response_template="<|start_header_id|>assistant<|end_header_id|>\n\n",
        )


def make_new_collator(tokenizer):
    """Span-based collator: assistant_turn with end_of_turn."""
    return TextCompletionsCollatorWithPadding(
        tokenizer=tokenizer,
        train_target="all_assistant_turns",
        response_template="<|start_header_id|>assistant<|end_header_id|>\n\n",
        end_of_turn_template="<|eot_id|>",
    )


old_collator = make_old_collator(tokenizer)
new_collator = make_new_collator(tokenizer)
print("Old collator train_target:", old_collator._default_collator.train_target)
print("New collator train_target:", new_collator._default_collator.train_target)

Old collator train_target: _legacy_instruction_response
New collator train_target: all_assistant_turns


/data/shanghong/oumi-pr2/src/oumi/core/collators/trl_data_collator_for_completion_only_lm.py:121: UserWarning: The pad_token_id and eos_token_id values of this tokenizer are identical. If you are planning for multi-turn training, it can result in the model continuously generating questions and answers without eos token. To avoid this, set the pad_token_id to a different value.
  warnings.warn(


In [6]:
from IPython.display import HTML, display


def visualize_masking(input_ids, labels, tokenizer, title=""):
    """Render tokens color-coded: green=unmasked (trained on), gray=masked."""
    html = f"<h4>{title}</h4><div style='font-family:monospace; line-height:1.8; white-space:pre-wrap;'>"
    for tid, lab in zip(input_ids, labels):
        tok_str = (
            tokenizer.decode([tid])
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace(" ", "\u00b7")
        )
        if lab == -100:
            html += f"<span style='color:#999; background:#f0f0f0;'>{tok_str}</span>"
        else:
            html += f"<span style='color:#000; background:#b5f5b5; font-weight:bold;'>{tok_str}</span>"
    html += "</div>"
    display(HTML(html))


def compare_masking(messages, tokenizer, old_collator, new_collator, title=""):
    """Tokenize messages, run both collators, show side-by-side."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    batch = [{"input_ids": input_ids}]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        old_out = old_collator(batch)
        new_out = new_collator(batch)

    old_labels = old_out["labels"][0].tolist()
    new_labels = new_out["labels"][0].tolist()
    ids = old_out["input_ids"][0].tolist()

    old_unmasked = sum(1 for x in old_labels if x != -100)
    new_unmasked = sum(1 for x in new_labels if x != -100)
    diffs = sum(1 for a, b in zip(old_labels, new_labels) if (a == -100) != (b == -100))

    print(f"=== {title} ===")
    print(f"Total tokens: {len(ids)}")
    print(
        f"Old unmasked: {old_unmasked}, New unmasked: {new_unmasked}, Diff positions: {diffs}"
    )

    visualize_masking(ids, old_labels, tokenizer, title=f"{title} — OLD (legacy)")
    visualize_masking(ids, new_labels, tokenizer, title=f"{title} — NEW (span-based)")

    if diffs > 0:
        print(f"\nDiffering tokens ({diffs}):")
        for i, (a, b) in enumerate(zip(old_labels, new_labels)):
            if (a == -100) != (b == -100):
                tok = tokenizer.decode([ids[i]])
                old_s = "TRAINED" if a != -100 else "masked"
                new_s = "TRAINED" if b != -100 else "masked"
                print(f"  pos {i}: {repr(tok):30s}  old={old_s:8s}  new={new_s}")
    return old_labels, new_labels

## 1. Toy Data

In [7]:
# Single-turn (no tool calls — should be similar)
toy_single = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
]
compare_masking(toy_single, tokenizer, old_collator, new_collator, "Toy: single-turn");

=== Toy: single-turn ===
Total tokens: 55
Old unmasked: 6, New unmasked: 6, Diff positions: 0


In [8]:
# Multi-turn (no tool calls)
toy_multi = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "That would be 6."},
]
compare_masking(toy_multi, tokenizer, old_collator, new_collator, "Toy: multi-turn");

=== Toy: multi-turn ===
Total tokens: 77
Old unmasked: 12, New unmasked: 13, Diff positions: 1



Differing tokens (1):
  pos 54: '<|eot_id|>'                    old=masked    new=TRAINED


In [9]:
# Tool-calling — THIS is where the legacy collator breaks!
# Llama formats tool results as <|start_header_id|>ipython<|end_header_id|>
# Legacy collator won't see this as an instruction boundary → trains on tool results
toy_tool = [
    {"role": "system", "content": "You are a helpful assistant with tool access."},
    {"role": "user", "content": "What's the weather in SF?"},
    {
        "role": "assistant",
        "content": '{"name": "get_weather", "arguments": {"city": "SF"}}',
    },
    {"role": "tool", "content": '{"temp": "65F", "condition": "sunny"}'},
    {"role": "assistant", "content": "It's 65F and sunny in San Francisco."},
]
compare_masking(
    toy_tool,
    tokenizer,
    old_collator,
    new_collator,
    "Toy: tool-calling (EXPECT BIG DIFF)",
);

=== Toy: tool-calling (EXPECT BIG DIFF) ===
Total tokens: 106
Old unmasked: 52, New unmasked: 28, Diff positions: 26



Differing tokens (26):
  pos 67: '<|eot_id|>'                    old=masked    new=TRAINED
  pos 68: '<|start_header_id|>'           old=TRAINED   new=masked
  pos 69: 'ipy'                           old=TRAINED   new=masked
  pos 70: 'thon'                          old=TRAINED   new=masked
  pos 71: '<|end_header_id|>'             old=TRAINED   new=masked
  pos 72: '\n\n'                          old=TRAINED   new=masked
  pos 73: '"{'                            old=TRAINED   new=masked
  pos 74: '\\"'                           old=TRAINED   new=masked
  pos 75: 'temp'                          old=TRAINED   new=masked
  pos 76: '\\":'                          old=TRAINED   new=masked
  pos 77: ' \\"'                          old=TRAINED   new=masked
  pos 78: '65'                            old=TRAINED   new=masked
  pos 79: 'F'                             old=TRAINED   new=masked
  pos 80: '\\",'                          old=TRAINED   new=masked
  pos 81: ' \\"'                     

In [10]:
# Multiple tool calls in sequence
toy_multi_tool = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Compare weather in SF and NYC"},
    {
        "role": "assistant",
        "content": '{"name": "get_weather", "arguments": {"city": "SF"}}',
    },
    {"role": "tool", "content": '{"temp": "65F", "condition": "sunny"}'},
    {
        "role": "assistant",
        "content": '{"name": "get_weather", "arguments": {"city": "NYC"}}',
    },
    {"role": "tool", "content": '{"temp": "45F", "condition": "cloudy"}'},
    {"role": "assistant", "content": "SF is 65F sunny, NYC is 45F cloudy."},
]
compare_masking(
    toy_multi_tool,
    tokenizer,
    old_collator,
    new_collator,
    "Toy: multi-tool (EXPECT BIGGER DIFF)",
);

=== Toy: multi-tool (EXPECT BIGGER DIFF) ===
Total tokens: 149
Old unmasked: 97, New unmasked: 49, Diff positions: 52



Differing tokens (52):
  pos 63: '<|eot_id|>'                    old=masked    new=TRAINED
  pos 64: '<|start_header_id|>'           old=TRAINED   new=masked
  pos 65: 'ipy'                           old=TRAINED   new=masked
  pos 66: 'thon'                          old=TRAINED   new=masked
  pos 67: '<|end_header_id|>'             old=TRAINED   new=masked
  pos 68: '\n\n'                          old=TRAINED   new=masked
  pos 69: '"{'                            old=TRAINED   new=masked
  pos 70: '\\"'                           old=TRAINED   new=masked
  pos 71: 'temp'                          old=TRAINED   new=masked
  pos 72: '\\":'                          old=TRAINED   new=masked
  pos 73: ' \\"'                          old=TRAINED   new=masked
  pos 74: '65'                            old=TRAINED   new=masked
  pos 75: 'F'                             old=TRAINED   new=masked
  pos 76: '\\",'                          old=TRAINED   new=masked
  pos 77: ' \\"'                     

## 2. TatQA Data

TatQA is single-turn (system/user/assistant, no tool calls), so diffs should be minimal.

In [11]:
TATQA_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/train_final_llama3.3_70b_instruct_max2048.jsonl"
with open(TATQA_PATH) as f:
    tatqa_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(tatqa_samples)} TatQA samples")

for i, sample in enumerate(tatqa_samples[:3]):
    msgs = sample["messages"]
    preview = msgs[-1]["content"][:80]
    print(f"\nSample {i}: {len(msgs)} messages, last assistant: {preview}...")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"TatQA sample {i}")

Loaded 5 TatQA samples

Sample 0: 3 messages, last assistant: <think> 
To determine the years in which the financial information was recorded,...
=== TatQA sample 0 ===
Total tokens: 382
Old unmasked: 104, New unmasked: 104, Diff positions: 0



Sample 1: 3 messages, last assistant: <think> To find the net debt receipts in 2019, we need to look at the Statement ...
=== TatQA sample 1 ===
Total tokens: 359
Old unmasked: 80, New unmasked: 80, Diff positions: 0



Sample 2: 3 messages, last assistant: <think> To find the net debt repayments in 2018, we need to look at the Statemen...
=== TatQA sample 2 ===
Total tokens: 349
Old unmasked: 69, New unmasked: 69, Diff positions: 0


## 3. Hermes Data

Hermes has tool-calling conversations. Llama maps `tool` → `ipython` role, which the legacy collator can't see as an instruction boundary. **Expect large masking differences**: legacy trains on tool results, span-based correctly masks them.

In [12]:
HERMES_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/hermes_reasoning_tool_use_train_split_train_filtered.jsonl"
with open(HERMES_PATH) as f:
    hermes_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(hermes_samples)} Hermes samples")

for i, sample in enumerate(hermes_samples[:3]):
    msgs = sample["messages"]
    roles = [m["role"] for m in msgs]
    print(f"\nSample {i}: {len(msgs)} messages, roles: {roles}")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"Hermes sample {i}")

Loaded 5 Hermes samples

Sample 0: 9 messages, roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'user', 'assistant', 'tool', 'assistant']
=== Hermes sample 0 ===
Total tokens: 1540
Old unmasked: 748, New unmasked: 414, Diff positions: 340



Differing tokens (340):
  pos 922: '<|eot_id|>'                    old=masked    new=TRAINED
  pos 923: '<|start_header_id|>'           old=TRAINED   new=masked
  pos 924: 'ipy'                           old=TRAINED   new=masked
  pos 925: 'thon'                          old=TRAINED   new=masked
  pos 926: '<|end_header_id|>'             old=TRAINED   new=masked
  pos 927: '\n\n'                          old=TRAINED   new=masked
  pos 928: '"<'                            old=TRAINED   new=masked
  pos 929: 'tool'                          old=TRAINED   new=masked
  pos 930: '_response'                     old=TRAINED   new=masked
  pos 931: '>\\'                           old=TRAINED   new=masked
  pos 932: 'n'                             old=TRAINED   new=masked
  pos 933: '{\\"'                          old=TRAINED   new=masked
  pos 934: 'tool'                          old=TRAINED   new=masked
  pos 935: '_call'                         old=TRAINED   new=masked
  pos 936: '_id'      


Differing tokens (142):
  pos 817: '<|eot_id|>'                    old=masked    new=TRAINED
  pos 818: '<|start_header_id|>'           old=TRAINED   new=masked
  pos 819: 'ipy'                           old=TRAINED   new=masked
  pos 820: 'thon'                          old=TRAINED   new=masked
  pos 821: '<|end_header_id|>'             old=TRAINED   new=masked
  pos 822: '\n\n'                          old=TRAINED   new=masked
  pos 823: '"<'                            old=TRAINED   new=masked
  pos 824: 'tool'                          old=TRAINED   new=masked
  pos 825: '_response'                     old=TRAINED   new=masked
  pos 826: '>\\'                           old=TRAINED   new=masked
  pos 827: 'n'                             old=TRAINED   new=masked
  pos 828: '{\\"'                          old=TRAINED   new=masked
  pos 829: 'tool'                          old=TRAINED   new=masked
  pos 830: '_call'                         old=TRAINED   new=masked
  pos 831: '_id'      

## Summary: Aggregate Diff Stats

In [13]:
def diff_stats(data_path, n=50):
    with open(data_path) as f:
        samples = [json.loads(line) for line in f.readlines()[:n]]
    total_diffs = 0
    total_tokens = 0
    total_old_unmasked = 0
    total_new_unmasked = 0
    legacy_trains_on_extra = 0  # tokens legacy trains on that span-based masks
    span_trains_on_extra = 0  # tokens span-based trains on that legacy masks
    for sample in samples:
        msgs = sample["messages"]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        input_ids = tokenizer.encode(text, add_special_tokens=False)
        batch = [{"input_ids": input_ids}]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            old_out = old_collator(batch)
            new_out = new_collator(batch)
        old_labels = old_out["labels"][0].tolist()
        new_labels = new_out["labels"][0].tolist()
        total_tokens += len(input_ids)
        total_old_unmasked += sum(1 for x in old_labels if x != -100)
        total_new_unmasked += sum(1 for x in new_labels if x != -100)
        for a, b in zip(old_labels, new_labels):
            if (a == -100) != (b == -100):
                total_diffs += 1
                if a != -100:  # legacy trains, span masks
                    legacy_trains_on_extra += 1
                else:  # span trains, legacy masks
                    span_trains_on_extra += 1
    print(f"  Samples: {len(samples)}")
    print(f"  Total tokens: {total_tokens}")
    print(f"  Old unmasked: {total_old_unmasked}")
    print(f"  New unmasked: {total_new_unmasked}")
    print(
        f"  Differing positions: {total_diffs} ({total_diffs / total_tokens * 100:.2f}%)"
    )
    print(f"    Legacy trains on (span masks): {legacy_trains_on_extra}")
    print(f"    Span trains on (legacy masks): {span_trains_on_extra}")


print("TatQA (no tool calls — expect small diff):")
diff_stats(TATQA_PATH)
print("\nHermes (has tool calls — expect large diff):")
diff_stats(HERMES_PATH)

TatQA (no tool calls — expect small diff):
  Samples: 50
  Total tokens: 33948
  Old unmasked: 7771
  New unmasked: 7771
  Differing positions: 0 (0.00%)
    Legacy trains on (span masks): 0
    Span trains on (legacy masks): 0

Hermes (has tool calls — expect large diff):
  Samples: 50
  Total tokens: 76246
  Old unmasked: 39373
  New unmasked: 26939
  Differing positions: 12634 (16.57%)
    Legacy trains on (span masks): 12534
    Span trains on (legacy masks): 100
